# BM25 training and development on Colab

Mount the project and supplied training files from Drive, then generate separate BM25 candidate samples for the frozen training and development Source 1 splits. The notebook can use CuPy for the sparse BM25 score multiplication when a CUDA GPU and matching CuPy wheel are available; otherwise it uses BM25S/Numba on CPU. Text preparation and top-k sorting remain on CPU.

This is a retrieval experiment, not a replacement for the submission pipeline. It reads only supplied training files and joins ground truth after retrieval to measure the sampled link recall. It does not read test files or use external business data. Outputs go under `artifacts/bm25_pipeline/{train,development}`.

In [ ]:
from pathlib import Path
import subprocess
import sys

from google.colab import drive
drive.mount('/content/drive')

# The Drive project should contain this repository and the supplied train TSVs.
PROJECT_DIR = Path('/content/drive/MyDrive/amazon-ml')
PACKAGE = PROJECT_DIR / 'code/business_entity_resolution'
TRAIN_DIR = PROJECT_DIR / 'student_resource/dataset/train'
ARTIFACTS_V2 = PACKAGE / 'artifacts/v2'

required = [TRAIN_DIR / name for name in (
    'train_source1.tsv', 'train_source2.tsv', 'train_source3.tsv', 'train_ground_truth.tsv'
)]
missing = [str(path) for path in required if not path.is_file()]
if missing:
    raise FileNotFoundError('Missing supplied training files:\n' + '\n'.join(missing))
if not (PACKAGE / 'bm25_pipeline/train.py').is_file():
    raise FileNotFoundError(f'BM25 project code not found under {PROJECT_DIR}')
print(f'Project: {PROJECT_DIR}')
print(f'Train data: {TRAIN_DIR}')

In [ ]:
# Use uv for reproducible dependency installation in the Colab environment.
subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', 'uv'], check=True)
subprocess.run(['uv', 'pip', 'install', '--system', '-r', str(PACKAGE / 'requirements.txt'),
                '-r', str(PACKAGE / 'requirements-retrieval.txt')], check=True)

# GPU is optional. Install the CUDA 12 wheel when the runtime exposes a GPU.
gpu_available = subprocess.run(['nvidia-smi', '-L'], capture_output=True, text=True).returncode == 0
if gpu_available:
    try:
        subprocess.run(['uv', 'pip', 'install', '--system', '-r',
                        str(PACKAGE / 'requirements-bm25-gpu.txt')], check=True)
        import cupy as cp
        cp.cuda.runtime.getDeviceCount()
        BACKEND = 'cupy'
        print('CuPy CUDA sparse scoring is available.')
    except Exception as exc:
        BACKEND = 'bm25s'
        print(f'CuPy could not access the GPU ({exc}); using BM25S/Numba CPU.')
else:
    BACKEND = 'bm25s'
    print('No CUDA GPU detected; using BM25S/Numba CPU.')
print('Selected backend:', BACKEND)

## Run settings

The defaults reproduce the pilot's bounded sample size and keep output volume manageable. Increase the per-country limits for a broader experiment. The saved split is the v2 seed-42 split; if no v2 DuckDB exists, the code imports the four supplied training files and creates the same split.

In [ ]:
DOCS_PER_COUNTRY = 150_000
TARGETS_PER_COUNTRY = 20_000
CANDIDATE_CAP = 32
BATCH_SIZE = 512

def run_partition(script_name):
    command = [
        sys.executable, str(PACKAGE / 'bm25_pipeline' / script_name),
        '--train-dir', str(TRAIN_DIR),
        '--artifacts-dir', str(ARTIFACTS_V2),
        '--backend', BACKEND,
        '--docs-per-country', str(DOCS_PER_COUNTRY),
        '--targets-per-country', str(TARGETS_PER_COUNTRY),
        '--cap', str(CANDIDATE_CAP),
        '--batch-size', str(BATCH_SIZE),
    ]
    subprocess.run(command, cwd=PROJECT_DIR, check=True)

run_partition('train.py')

In [ ]:
run_partition('develop.py')

## Archive this run and train XGBoost

This creates a timestamped run folder, copies the BM25 reports and compressed candidate files into it, then runs the existing v2 XGBoost training pipeline on its existing candidate/features pipeline. XGBoost performs its held-out final validation during training. The separate report command below renders those metrics.

In [ ]:
from datetime import datetime, timezone
import json
import shutil

RUN_ID = datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')
RUN_DIR = PACKAGE / 'artifacts/bm25_pipeline/runs' / RUN_ID
BM25_ARCHIVE = RUN_DIR / 'bm25_reports'
XGBOOST_ARCHIVE = RUN_DIR / 'xgboost'
BM25_ARCHIVE.mkdir(parents=True, exist_ok=True)
XGBOOST_ARCHIVE.mkdir(parents=True, exist_ok=True)

BM25_FILES = {}
for partition in ('train', 'development'):
    report = PACKAGE / 'artifacts/bm25_pipeline' / partition / f'{"training" if partition == "train" else "development"}_report_{BACKEND}.json'
    candidates = PACKAGE / 'artifacts/bm25_pipeline' / partition / f'{"training" if partition == "train" else "development"}_candidates_{BACKEND}.tsv.gz'
    if not report.is_file() or not candidates.is_file():
        raise FileNotFoundError(f'BM25 {partition} outputs are incomplete; run that split first.')
    archived_report = BM25_ARCHIVE / report.name
    archived_candidates = BM25_ARCHIVE / candidates.name
    shutil.copy2(report, archived_report)
    shutil.copy2(candidates, archived_candidates)
    BM25_FILES[partition] = {'report': str(archived_report), 'candidates': str(archived_candidates),
                          'original_candidates': str(candidates)}
print(f'Run archive: {RUN_DIR}')
print('BM25 outputs:', json.dumps(BM25_FILES, indent=2))

In [ ]:
# Keep the requested XGBoost version and TensorBoard event writer available.
subprocess.run(['uv', 'pip', 'install', '--system', 'xgboost==3.4.1', 'tensorboard==2.21.0'], check=True)
XGBOOST_ARTIFACTS = ARTIFACTS_V2 / 'models/xgboost'
XGBOOST_ARTIFACTS.mkdir(parents=True, exist_ok=True)
training_log = XGBOOST_ARCHIVE / 'xgboost_training.log'
command = [
    sys.executable, str(PACKAGE / 'src/train_v2.py'),
    '--train-dir', str(TRAIN_DIR), '--artifacts-dir', str(ARTIFACTS_V2),
    '--model', 'xgboost',
]
print('Starting v2 XGBoost training and held-out validation...')
with training_log.open('w', encoding='utf-8') as log_file:
    process = subprocess.Popen(command, cwd=PROJECT_DIR, stdout=subprocess.PIPE,
                               stderr=subprocess.STDOUT, text=True, bufsize=1)
    for line in process.stdout:
        print(line, end='')
        log_file.write(line)
        log_file.flush()
    exit_code = process.wait()
if exit_code:
    raise subprocess.CalledProcessError(exit_code, command)
print(f'XGBoost log saved to {training_log}')

## Render held-out validation report and save artifacts

The XGBoost training script writes final-validation metrics and model checkpoints under the shared v2 artifact tree. This cell renders the held-out report and snapshots the final model, configuration, metrics, report, and BM25 reports in the timestamped run folder. It does not run test inference or use test labels.

In [ ]:
report_path = XGBOOST_ARCHIVE / 'validation_report.md'
report_command = [
    sys.executable, str(PACKAGE / 'src/report_v2.py'),
    '--artifacts-dir', str(ARTIFACTS_V2), '--model', 'xgboost',
    '--output', str(report_path),
]
subprocess.run(report_command, cwd=PROJECT_DIR, check=True)
for name in ('matcher_model.joblib', 'model_config.json', 'training_metrics.json'):
    source = XGBOOST_ARTIFACTS / name
    if not source.is_file():
        raise FileNotFoundError(f'Missing XGBoost output: {source}')
    shutil.copy2(source, XGBOOST_ARCHIVE / name)

manifest = {
    'run_id': RUN_ID,
    'bm25_backend': BACKEND,
    'bm25_outputs': BM25_FILES,
    'xgboost_model_artifacts': str(XGBOOST_ARTIFACTS),
    'xgboost_checkpoint_directory': str(XGBOOST_ARTIFACTS / 'work/model_checkpoints'),
    'xgboost_archive': str(XGBOOST_ARCHIVE),
    'validation_report': str(report_path),
    'test_labels_used': False,
}
(RUN_DIR / 'run_manifest.json').write_text(json.dumps(manifest, indent=2), encoding='utf-8')
print(report_path.read_text(encoding='utf-8'))
print(f'Archived XGBoost model, config, metrics, report, and log under {XGBOOST_ARCHIVE}')

## Results

Each run writes a compressed candidate TSV and JSON report to its own `train/` or `development/` directory. The reports include the selected backend, per-country elapsed time, candidate volume, source-specific positive-link recall, and the fraction of sampled linked Source 1 entities for which every sampled true link was retained. Development metrics are descriptive for the fixed sample; they are not the official held-out model F₀.₅.